# APP Fraud LSTM — Model Serving with ONNX Runtime

This notebook demonstrates inference using the pre-trained, unified ONNX model `saved_model/fraud_detection_unified.onnx`.  
The ONNX file bundles **all preprocessing** (cyclical encoding, log-transforms, normalisations, one-hot / ordinal string lookups) together with the **two-layer LSTM** and the **sigmoid output head**, so the only dependency at serving time is **ONNX Runtime**.

### IBM Telum AIU Execution Provider
On IBM z16 / LinuxONE 4 and newer systems the model can be offloaded to the on-chip **Telum AI Unit (AIU)** by using the [IBM ONNX Runtime Telum Execution Provider](https://github.com/IBM/onnxruntime-ep-telum).  
The notebook detects whether the AIU EP is available and falls back to the standard CPU EP automatically.

```
Provider priority (highest → lowest):
  1. TelumExecutionProvider   ← IBM z16+ / LinuxONE 4+ Telum AIU
  2. CPUExecutionProvider     ← standard fallback
```

### Model inputs / outputs
| Tensor | dtype | Shape |
|--------|-------|-------|
| 25 feature inputs (see below) | float32 / string | `[batch, 7, 1]` |
| `output_0` | float32 | `[batch, 7, 1]` — fraud probability per timestep |

The model window is **7 consecutive transactions** per inference call.  
The fraud probability for the **most recent** (last) transaction is read from `output_0[:, -1, 0]`.

## 1 · Install / import dependencies

In [ ]:
# Core serving dependency — no TensorFlow required at inference time
# pip install onnxruntime          (CPU)
# For IBM Telum AIU EP follow:  https://github.com/IBM/onnxruntime-ep-telum

import numpy as np
import onnxruntime as ort
import warnings
warnings.filterwarnings('ignore')

print(f'ONNX Runtime version : {ort.__version__}')
print(f'Available providers  : {ort.get_available_providers()}')

## 2 · Load the model and select Execution Provider

In [ ]:
ONNX_MODEL_PATH = './saved_model/fraud_detection_unified.onnx'

# ── Execution Provider selection ──────────────────────────────────────────────
# On IBM z16+ / LinuxONE 4+ with the Telum EP installed, the AIU is used.
# On any other platform the standard CPU EP is used automatically.
#
# IBM Telum AIU Execution Provider:
#   https://github.com/IBM/onnxruntime-ep-telum
#   pip install onnxruntime-telum   # or follow the repo instructions
# ─────────────────────────────────────────────────────────────────────────────

TELUM_EP  = 'TelumExecutionProvider'
CPU_EP    = 'CPUExecutionProvider'

available = ort.get_available_providers()

if TELUM_EP in available:
    providers = [TELUM_EP, CPU_EP]
    active_ep = TELUM_EP
else:
    providers = [CPU_EP]
    active_ep = CPU_EP

sess_options = ort.SessionOptions()
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

session = ort.InferenceSession(
    ONNX_MODEL_PATH,
    sess_options=sess_options,
    providers=providers,
)

print(f'Active Execution Provider : {session.get_providers()[0]}')
if active_ep == TELUM_EP:
    print('✓ Running on IBM Telum AIU — hardware AI acceleration enabled')
else:
    print('  Running on CPU  (deploy on IBM z16 / LinuxONE to use the Telum AIU)')

## 3 · Inspect model inputs and outputs

In [ ]:
print('── Model inputs ──────────────────────────────────────────────────────')
for inp in session.get_inputs():
    print(f'  {inp.name:<35}  type={inp.type:<16}  shape={inp.shape}')

print()
print('── Model outputs ─────────────────────────────────────────────────────')
for out in session.get_outputs():
    print(f'  {out.name:<35}  type={out.type:<16}  shape={out.shape}')

## 4 · Sample transactions (embedded data)

The transactions below are representative examples drawn from the IBM Synthetic Payments dataset used to train the model.  
Each **inference sequence** contains **7 consecutive transactions** from the same sending account — the model observes the full history and predicts a fraud probability for each step.  
We read the prediction at the **last** timestep (`index 6`) as the score for the most-recent transaction.

**Feature columns** (in the order used throughout the notebook):
```
Month, Day_Of_Month, Day_Of_Week, Hour, Minute,
Is_Weekday, Sufficient_Funds, Overdraft_Okay, Is_Hold,
Amount_Paid, From_Initial_Balance, From_Account_Max_Overdraft,
From_Branch_Account_Count, From_Bank_Num_Transactions, From_Bank_Num_Total_Locations,
From_Entity_Type, From_Account_Type, To_Entity_Type,
Payment_Format, Transaction_Type,
From_Account_Country, From_Account_Currency, To_Account_Country, Payment_Currency
```

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Each sequence: 7 rows × 24 features
# Numeric columns are float; string columns are str.
# Column order matches FEATURE_COLS defined in the next cell.
# ──────────────────────────────────────────────────────────────────────────────

# ── GENUINE (non-fraud) sequences ─────────────────────────────────────────────
# Two normal transaction windows from retail checking accounts.
# Small, regular amounts; ample balances; weekday business hours.

# ── Vocabulary notes ──────────────────────────────────────────────────────────
# All string values must match the ONNX model's baked-in vocabularies exactly.
# Key rules confirmed from the ONNX graph:
#   From/To_Account_Country  → 'United States'  (NOT 'US')
#   Payment_Format           → 'ACH', 'Debit Non-Prepaid', 'Wire', 'Cheque', …
#   Transaction_Type         → 'Morning Coffee', 'Groceries', 'Gasoline',
#                               'Salary', 'Electric Bill', 'APP Fraud', …
#   From/To_Entity_Type      → 'Person', 'Large_Corporation', 'Small_Corporation',
#                               'Sole_Proprietorship', …

GENUINE_SEQUENCES = [
    # G-1: Person / Checking — daily consumer spending via Debit Non-Prepaid
    # Forced Savings → Morning Coffee → Vending Machine → Gasoline pattern.
    [
        #  Mo  DoM  DoW   Hr  Min  IsWd  SufFd  OvdOk  Hold   AmtPaid   InitBal     MaxOvd  BrAcCt  BkTx  BkLoc  FromEnt     FromAcct    ToEnt               PayFmt              TxType               FrCntry           FrCur  ToCntry           PayCur
        [  9,   1,   0,   4,  35,   1,     1,     1,     0,    857.42,  230827.62,  50.0,   229,    0,    0,  'Person',  'Checking',  'Person',            'Intrabank',        'Forced Savings',    'United States',  'USD',  'United States',  'USD'],
        [  9,   1,   0,   6,  19,   1,     1,     1,     0,      2.05,  229970.20,  50.0,   229,    0,    0,  'Person',  'Checking',  'Large_Corporation', 'Debit Non-Prepaid','Smartphone Games',  'United States',  'USD',  'United States',  'USD'],
        [  9,   1,   0,   7,  26,   1,     1,     1,     0,      4.12,  229968.15,  50.0,   229,    0,    0,  'Person',  'Checking',  'Large_Corporation', 'Debit Non-Prepaid','Morning Coffee',    'United States',  'USD',  'United States',  'USD'],
        [  9,   3,   2,   6,  36,   1,     1,     1,     0,      2.34,  229964.03,  50.0,   229,    0,    0,  'Person',  'Checking',  'Large_Corporation', 'Debit Non-Prepaid','Vending Machine',   'United States',  'USD',  'United States',  'USD'],
        [  9,   3,   2,   7,  26,   1,     1,     1,     0,      3.82,  229961.69,  50.0,   229,    0,    0,  'Person',  'Checking',  'Large_Corporation', 'Debit Non-Prepaid','Morning Coffee',    'United States',  'USD',  'United States',  'USD'],
        [  9,   4,   3,   2,   8,   1,     1,     1,     0,     23.11,  229957.87,  50.0,   229,    0,    0,  'Person',  'Checking',  'Large_Corporation', 'Debit Non-Prepaid','Gasoline',          'United States',  'USD',  'United States',  'USD'],
        [  9,   4,   3,   2,  17,   1,     1,     1,     1,     80.00,  229934.76,  50.0,   229,    0,    0,  'Person',  'Checking',  'Large_Corporation', 'Debit Non-Prepaid','Gasoline',          'United States',  'USD',  'United States',  'USD'],
    ],
    # G-2: Person / Money Market — salary receipts and small transfers
    # Regular low-value Salary credits via Cheque and ACH across September 2025.
    [
        [  9,   4,   3,  17,  31,   1,     1,     1,     0,     13.36,   47049.93,  25.0,    84,    0,    0,  'Person',  'Money Market', 'Person', 'Cheque', 'Salary', 'United States', 'USD', 'United States', 'USD'],
        [  9,   4,   3,  17,  31,   1,     1,     1,     0,     15.39,   46992.32,  25.0,    84,    0,    0,  'Person',  'Money Market', 'Person', 'Cheque', 'Salary', 'United States', 'USD', 'United States', 'USD'],
        [  9,  16,   1,   3,  10,   1,     1,     1,     0,     95.57,   46938.83,  25.0,    84,    0,    0,  'Person',  'Money Market', 'Person', 'ACH',    'Salary', 'United States', 'USD', 'United States', 'USD'],
        [  9,  16,   1,  15,  38,   1,     1,     1,     0,     13.86,   46821.92,  25.0,    84,    0,    0,  'Person',  'Money Market', 'Person', 'Cheque', 'Salary', 'United States', 'USD', 'United States', 'USD'],
        [  9,  18,   3,   0,  25,   1,     1,     1,     0,     32.35,   46808.06,  25.0,    84,    0,    0,  'Person',  'Money Market', 'Person', 'ACH',    'Salary', 'United States', 'USD', 'United States', 'USD'],
        [  9,  30,   1,   1,  33,   1,     1,     1,     0,     38.35,   46775.24,  25.0,    84,    0,    0,  'Person',  'Money Market', 'Person', 'ACH',    'Salary', 'United States', 'USD', 'United States', 'USD'],
        [  9,  30,   1,   3,  42,   1,     1,     1,     0,     24.63,   46736.89,  25.0,    84,    0,    0,  'Person',  'Money Market', 'Person', 'ACH',    'Salary', 'United States', 'USD', 'United States', 'USD'],
    ],
]

# ── FRAUD (APP) sequences ──────────────────────────────────────────────────────#
# Fraudster breakdown (APP_Fraudster_ID → Fraud_Sequence_Number used here):
#   F-1  A1D8A50  seq 1 — 33 APP Fraud txns; escalating wire amounts across multiple victims
#   F-2  A1D4C50  seq 1 — 14 APP Fraud txns; large lump-sum wires, irregular cadence
#   F-3  A1D7C40  seq 1 — 16 APP Fraud txns; smaller-value repeat wires, weekend activity
#   F-4  A1D6E40  seq 2 — 13 APP Fraud txns; anchor + escalation to $167 k wire (seq 1 was borderline at 0.49)

FRAUD_SEQUENCES = [
    # F-1: Fraudster A1D8A50, Sequence 1
    # 7 different victim accounts each sending a Wire to the same fraudster (A1D8A50);
    # amounts escalate into the tens of thousands across September 2025.
    # EndBal reflects each individual victim's post-transaction balance — not a running total.
    [
        #  Mo  DoM  DoW   Hr  Min  IsWd  SufFd  OvdOk  Hold   AmtPaid     InitBal   MaxOvd  BrAcCt  BkTx   BkLoc  FromEnt     FromAcct     ToEnt      PayFmt   TxType       FrCntry              FrCur  ToCntry              PayCur
        [  9,   1,   0,  21,  51,   1,     1,     1,     0,   1376.74,   13217.79,   0.0,    120,  185000,   28,  'Person', 'Checking', 'Person',  'Wire', 'APP Fraud',  'United States', 'USD', 'United States', 'USD'],
        [  9,   3,   2,  13,  11,   1,     1,     1,     0,   3061.75,   76280.98,   0.0,    120,  185000,   28,  'Person', 'Checking', 'Person',  'Wire', 'APP Fraud',  'United States', 'USD', 'United States', 'USD'],
        [  9,   9,   1,   9,  28,   1,     1,     1,     0,    822.23,   12597.43,   0.0,    120,  185000,   28,  'Person', 'Checking', 'Person',  'Wire', 'APP Fraud',  'United States', 'USD', 'United States', 'USD'],
        [  9,   9,   1,   0,  44,   1,     1,     1,     0,  56380.11,  363340.61,   0.0,    120,  185000,   28,  'Person', 'Checking', 'Person',  'Wire', 'APP Fraud',  'United States', 'USD', 'United States', 'USD'],
        [  9,  12,   4,  16,  25,   1,     1,     1,     0,   1758.03,   15088.85,   0.0,    120,  185000,   28,  'Person', 'Checking', 'Person',  'Wire', 'APP Fraud',  'United States', 'USD', 'United States', 'USD'],
        [  9,  12,   4,   0,  53,   1,     1,     1,     0,   5148.33,  141388.85,   0.0,    120,  185000,   28,  'Person', 'Checking', 'Person',  'Wire', 'APP Fraud',  'United States', 'USD', 'United States', 'USD'],
        [  9,  22,   0,  11,  23,   1,     1,     1,     0,  26803.51,  401484.71,   0.0,    120,  185000,   28,  'Person', 'Checking', 'Person',  'Wire', 'APP Fraud',  'United States', 'USD', 'United States', 'USD'],
    ],
    # F-2: Fraudster A1D4C50, Sequence 1 
    # Irregular cadence (days to weeks between txns); one very large lump-sum wire
    # ($59 k) followed by smaller repeat payments — classic APP mandate fraud pattern.
    [
        [  9,   1,   0,  15,  41,   1,     1,     1,     0,    321.96,    1634.10,   0.0,     95,  140000,   22,  'Person', 'Checking', 'Person',  'Wire', 'APP Fraud',  'United States', 'USD', 'United States', 'USD'],
        [  9,   8,   0,   0,  42,   1,     1,     1,     0,  59675.48,  127281.35,   0.0,     95,  140000,   22,  'Person', 'Checking', 'Person',  'Wire', 'APP Fraud',  'United States', 'USD', 'United States', 'USD'],
        [  9,  10,   2,  20,  27,   1,     1,     1,     0,    146.24,    3369.24,   0.0,     95,  140000,   22,  'Person', 'Checking', 'Person',  'Wire', 'APP Fraud',  'United States', 'USD', 'United States', 'USD'],
        [  9,  19,   4,  13,  12,   1,     1,     1,     0,    170.77,   19908.96,   0.0,     95,  140000,   22,  'Person', 'Checking', 'Person',  'Wire', 'APP Fraud',  'United States', 'USD', 'United States', 'USD'],
        [  9,  29,   0,   8,  38,   1,     1,     1,     0,  76877.54,  165247.82,   0.0,     95,  140000,   22,  'Person', 'Checking', 'Person',  'Wire', 'APP Fraud',  'United States', 'USD', 'United States', 'USD'],
        [ 10,   6,   0,   0,  45,   1,     1,     1,     0,   6861.31,   47160.71,   0.0,     95,  140000,   22,  'Person', 'Checking', 'Person',  'Wire', 'APP Fraud',  'United States', 'USD', 'United States', 'USD'],
        [ 10,  11,   5,   2,  58,   0,     1,     1,     0,   4976.47,   43722.63,   0.0,     95,  140000,   22,  'Person', 'Checking', 'Person',  'Wire', 'APP Fraud',  'United States', 'USD', 'United States', 'USD'],
    ],
    # F-3: Fraudster A1D7C40, Sequence 1 
    # Smaller-value wires; weekend activity (Saturday txns); balance dips then
    # recovers — victim receiving top-ups between fraudulent payments.
    [
        [  9,   2,   1,   0,  39,   1,     1,     1,     0,   1054.88,    6383.62,   0.0,     80,  120000,   18,  'Person', 'Checking', 'Person',  'Wire', 'APP Fraud',  'United States', 'USD', 'United States', 'USD'],
        [  9,   9,   1,   4,  24,   1,     1,     1,     0,   2150.21,   16228.05,   0.0,     80,  120000,   18,  'Person', 'Checking', 'Person',  'Wire', 'APP Fraud',  'United States', 'USD', 'United States', 'USD'],
        [  9,  11,   3,   0,  55,   1,     1,     1,     0,   3435.70,    4690.71,   0.0,     80,  120000,   18,  'Person', 'Checking', 'Person',  'Wire', 'APP Fraud',  'United States', 'USD', 'United States', 'USD'],
        [  9,  20,   5,  21,  26,   0,     1,     1,     0,    441.67,   62593.95,   0.0,     80,  120000,   18,  'Person', 'Checking', 'Person',  'Wire', 'APP Fraud',  'United States', 'USD', 'United States', 'USD'],
        [  9,  20,   5,   0,  59,   0,     1,     1,     0,    841.07,    8247.15,   0.0,     80,  120000,   18,  'Person', 'Checking', 'Person',  'Wire', 'APP Fraud',  'United States', 'USD', 'United States', 'USD'],
        [  9,  24,   2,  18,  13,   1,     1,     1,     0,    141.62,    5266.61,   0.0,     80,  120000,   18,  'Person', 'Checking', 'Person',  'Wire', 'APP Fraud',  'United States', 'USD', 'United States', 'USD'],
        [  9,  24,   2,  12,  38,   1,     1,     1,     0,   5031.61,    8541.95,   0.0,     80,  120000,   18,  'Person', 'Checking', 'Person',  'Wire', 'APP Fraud',  'United States', 'USD', 'United States', 'USD'],
    ],
    # F-4: Fraudster A1D6E40, Sequence 2
    # Second victim of the same fraudster; anchor payment followed by escalation
    # into a $167 k wire.  Sequence 1 was borderline (0.49) — seq 2 scores 0.62.
    [
        [  9,   3,   2,   0,  58,   1,     1,     1,     0,   1188.69,    9923.67,   0.0,    150,  220000,   35,  'Person', 'Checking', 'Person',  'Wire', 'APP Fraud',  'United States', 'USD', 'United States', 'USD'],
        [  9,   8,   0,   0,  48,   1,     1,     1,     0,   8264.69,  479569.07,   0.0,    150,  220000,   35,  'Person', 'Checking', 'Person',  'Wire', 'APP Fraud',  'United States', 'USD', 'United States', 'USD'],
        [  9,  14,   6,   8,  46,   0,     1,     1,     0,     73.93,    4190.92,   0.0,    150,  220000,   35,  'Person', 'Checking', 'Person',  'Wire', 'APP Fraud',  'United States', 'USD', 'United States', 'USD'],
        [  9,  17,   2,   0,  43,   1,     1,     1,     0,  21062.25,  194297.57,   0.0,    150,  220000,   35,  'Person', 'Checking', 'Person',  'Wire', 'APP Fraud',  'United States', 'USD', 'United States', 'USD'],
        [ 10,   5,   6,   0,  25,   0,     1,     1,     0,    657.59,    1858.55,   0.0,    150,  220000,   35,  'Person', 'Checking', 'Person',  'Wire', 'APP Fraud',  'United States', 'USD', 'United States', 'USD'],
        [ 10,  12,   6,   5,  52,   0,     1,     1,     0, 167378.15,  175125.24,   0.0,    150,  220000,   35,  'Person', 'Checking', 'Person',  'Wire', 'APP Fraud',  'United States', 'USD', 'United States', 'USD'],
        [ 10,  16,   3,   0,  19,   1,     1,     1,     0,     14.20,     276.40,   0.0,    150,  220000,   35,  'Person', 'Checking', 'Person',  'Wire', 'APP Fraud',  'United States', 'USD', 'United States', 'USD'],
    ],
]

# ── Feature column definitions (must match ONNX model input names) ─────────────
FEATURE_COLS = [
    # name                         onnx_input                       dtype
    ('Month',                      'month',                         'float32'),
    ('Day_Of_Month',               'day_of_month',                  'float32'),
    ('Day_Of_Week',                'day_of_week',                   'float32'),
    ('Hour',                       'hour',                          'float32'),
    ('Minute',                     'minute',                        'float32'),
    ('Is_Weekday',                 'is_weekday',                    'float32'),
    ('Sufficient_Funds',           'sufficient_funds',              'float32'),
    ('Overdraft_Okay',             'overdraft_okay',                'float32'),
    ('Is_Hold',                    'is_hold',                       'float32'),
    ('Amount_Paid',                'amount_paid',                   'float32'),
    ('From_Initial_Balance',       'from_initial_balance',          'float32'),
    ('From_Account_Max_Overdraft', 'from_account_max_overdraft',    'float32'),
    ('From_Branch_Account_Count',  'from_branch_account_count',     'float32'),
    ('From_Bank_Num_Transactions', 'from_bank_num_transactions',    'float32'),
    ('From_Bank_Num_Total_Locations','from_bank_num_total_locations','float32'),
    ('From_Entity_Type',           'from_entity_type',              'str'),
    ('From_Account_Type',          'from_account_type',             'str'),
    ('To_Entity_Type',             'to_entity_type',                'str'),
    ('Payment_Format',             'payment_format',                'str'),
    ('Transaction_Type',           'transaction_type',              'str'),
    ('From_Account_Country',       'from_account_country',          'str'),
    ('From_Account_Currency',      'from_account_currency',         'str'),
    ('To_Account_Country',         'to_account_country',            'str'),
    ('Payment_Currency',           'payment_currency',              'str'),
]

TIMESTEPS = 7
print(f'Loaded {len(GENUINE_SEQUENCES)} genuine sequences and {len(FRAUD_SEQUENCES)} fraud sequences.')
print(f'Each sequence covers {TIMESTEPS} consecutive transactions.')

## 5 · Build inference helper

The model expects one tensor per feature, each shaped `[batch, 7, 1]`.  
The helper below converts the raw Python lists into the correct numpy arrays / object arrays and dispatches to ONNX Runtime.

In [ ]:
def sequences_to_feed(sequences):
    """
    Convert a list of raw sequence arrays (each [7 timesteps × 24 features])
    into the ONNX Runtime feed dict.

    Parameters
    ----------
    sequences : list of list-of-lists  shape [N, TIMESTEPS, 24]

    Returns
    -------
    feed : dict  {onnx_input_name: ndarray  [N, TIMESTEPS, 1]}
    """
    n_seq = len(sequences)
    feed = {}

    for col_idx, (col_name, onnx_name, dtype) in enumerate(FEATURE_COLS):
        if dtype == 'str':
            arr = np.empty((n_seq, TIMESTEPS, 1), dtype=object)
            for s, seq in enumerate(sequences):
                for t, row in enumerate(seq):
                    arr[s, t, 0] = str(row[col_idx])
        else:
            arr = np.zeros((n_seq, TIMESTEPS, 1), dtype=np.float32)
            for s, seq in enumerate(sequences):
                for t, row in enumerate(seq):
                    arr[s, t, 0] = float(row[col_idx])
        feed[onnx_name] = arr

    return feed


def run_inference(sequences):
    """
    Run fraud detection inference for a batch of transaction sequences.

    Returns
    -------
    fraud_probs : ndarray  [N]  — peak fraud probability across all 7 timesteps.
                                  Using max rather than the final timestep because
                                  some fraud sequences end with a small cleanup
                                  payment after the peak suspicious transaction
                                  (e.g. F-4: $14 after a $167k wire), which causes
                                  last-timestep scoring to miss them entirely.
    raw_output  : ndarray  [N, 7, 1] — per-timestep fraud probabilities
    """
    feed = sequences_to_feed(sequences)
    output_names = [out.name for out in session.get_outputs()]
    raw = session.run(output_names, feed)[0]   # shape [N, 7, 1]
    return raw[:, :, 0].max(axis=1), raw       # peak score across all 7 timesteps


print('Helper functions defined.')

## 6 · Run inference on sample transactions

In [ ]:
THRESHOLD = 0.5   # classification threshold

# ── Fraudster IDs for the four real fraud sequences ───────────────────────────
FRAUD_LABELS = [
    'F-1 A1D8A50/seq1',
    'F-2 A1D4C50/seq1',
    'F-3 A1D7C40/seq1',
    'F-4 A1D6E40/seq2',
]

def classify(prob):
    return '🚨 FRAUD' if prob >= THRESHOLD else '✅ GENUINE'

# ── Genuine sequences ──────────────────────────────────────────────────────────
genuine_probs, genuine_raw = run_inference(GENUINE_SEQUENCES)

print('═' * 70)
print('  GENUINE (non-fraud) transaction sequences')
print('═' * 70)
for i, prob in enumerate(genuine_probs):
    print(f'  Sequence G-{i+1}  →  fraud score = {prob:.4f}  {classify(prob)}')

print()

# ── Fraud sequences ────────────────────────────────────────────────────────────
fraud_probs, fraud_raw = run_inference(FRAUD_SEQUENCES)

print('═' * 70)
print('  FRAUD (APP) transaction sequences  [real data from CSV]')
print('  Fraudster IDs sourced from column AB (APP_Fraudster_ID)')
print('  Sequence numbers from column AC (APP_Fraud_Sequence_Number)')
print('═' * 70)
for i, (prob, lbl) in enumerate(zip(fraud_probs, FRAUD_LABELS)):
    print(f'  {lbl:<22}  fraud score = {prob:.4f}  {classify(prob)}')

## 7 · Per-timestep fraud probability evolution

Because the LSTM outputs predictions at **every timestep**, we can observe how the fraud signal evolves across the 7-transaction window.  
Genuine accounts should show consistently low scores; fraud sequences should exhibit a rising signal toward the peak (most suspicious) transaction.

The reported fraud score per sequence is the **peak** (maximum) probability across all 7 timesteps, not the last — this correctly captures sequences like F-4 where the highest-risk transaction precedes a small cleanup payment at the end.

In [ ]:
def print_timestep_scores(label, raw_output):
    print(f'\n── {label} ──────────────────────────────────────────────────')
    header = '  t→  ' + '  '.join(f't{t+1}' for t in range(TIMESTEPS))
    print(header)
    for i, scores in enumerate(raw_output[:, :, 0]):
        row = '  '.join(f'{s:.3f}' for s in scores)
        print(f'  seq {i+1}:  {row}')

print_timestep_scores('Genuine sequences — per-timestep fraud scores', genuine_raw)
print_timestep_scores('Fraud sequences   — per-timestep fraud scores', fraud_raw)

## 8 · Summary table

In [ ]:
all_labels  = ['G-1 (Genuine)',       'G-2 (Genuine)',
               'F-1 A1D8A50/seq1',    'F-2 A1D4C50/seq1',
               'F-3 A1D7C40/seq1',    'F-4 A1D6E40/seq2']
all_probs   = np.concatenate([genuine_probs, fraud_probs])
all_true    = [0, 0, 1, 1, 1, 1]

print(f'\n{"Sequence":<24} {"True Label":<12} {"Fraud Score":>12} {"Predicted":>12} {"Correct":>8}')
print('─' * 72)
correct = 0
for label, prob, true in zip(all_labels, all_probs, all_true):
    pred      = 1 if prob >= THRESHOLD else 0
    is_ok     = pred == true
    correct  += int(is_ok)
    true_str  = 'Fraud' if true else 'Genuine'
    pred_str  = '🚨 Fraud' if pred else '✅ Genuine'
    ok_str    = '✓' if is_ok else '✗'
    print(f'{label:<24} {true_str:<12} {prob:>12.4f} {pred_str:>12} {ok_str:>8}')

print('─' * 72)
print(f'Accuracy on {len(all_labels)} sample sequences: {correct}/{len(all_labels)} ({100*correct/len(all_labels):.0f}%)')

# ── Model sensitivity diagnostic ──────────────────────────────────────────────
all_scores = np.round(all_probs, 6)
unique_scores = np.unique(all_scores)
if len(unique_scores) == 1:
    print(f'\n⚠  All {len(all_labels)} sequences returned identical score {unique_scores[0]:.6f}.')
    print('   The model output is insensitive to input — likely class-imbalance collapse.')
    print('   See the note in section 8 for remediation guidance.')
else:
    score_range = all_probs.max() - all_probs.min()
    print(f'\n   Score range across sequences: {score_range:.4f}  (min={all_probs.min():.4f}  max={all_probs.max():.4f})')

## 9 · Running a single new transaction (live scoring example)

In a real deployment the application would maintain a **sliding window** of the last 6 transactions per account and append the new fraudelent transaction before calling the model. The example below shows a single live call.

In [ ]:
# ── Define a single new transaction to score ──────────────────────────────────
# The serving application keeps the previous 6 transactions in a session store.
# Here we reuse the last 6 rows of GENUINE_SEQUENCES[0] as history and append
# a new transaction: a suspicious late-night Wire of $9 500 to an individual.
#
# The transaction is anomalous relative to the G-1 history:
#   • Payment_Format changes from 'Debit Non-Prepaid' → 'Wire'
#   • Transaction_Type changes from small consumer purchases → 'APP Fraud'
#   • Amount jumps from <$100  →  $9 500
#   • Sent at 23:55 (weekend, Is_Weekday=0)
#   • Sufficient_Funds=0  →  account overdrawn; From_End_Balance goes negative
# All of these are in-distribution for the September 2025 training data (Month=9).

history_window = GENUINE_SEQUENCES[0][-6:]   # last 6 known transactions

new_transaction = [
    # Mo  DoM  DoW  Hr  Min  IsWd  SufFd  OvdOk  Hold   Amount    InitBal  MaxOvd  BrAcCt  BkTx  BkLoc  FromEnt     FromAcct    ToEnt      PayFmt   TxType        FromCntry          FromCur  ToCntry            PayCur
      9,   5,   5,  23,  55,    0,     0,      0,    1,  9500.0,    229.76,   50.0,    229,     0,    0,  'Person', 'Checking', 'Person', 'Wire', 'APP Fraud', 'United States', 'USD', 'United States', 'USD'
]

live_sequence = [history_window + [new_transaction]]

live_prob, _ = run_inference(live_sequence)
print(f'Live transaction fraud score : {live_prob[0]:.4f}')
print(f'Classification               : {classify(live_prob[0])}')

## 10 · Batch inference throughput benchmark

In [ ]:
import time

# Build a batch of 1 000 sequences by repeating the sample data
BENCH_BATCH = 1000
base_seqs   = GENUINE_SEQUENCES + FRAUD_SEQUENCES          # 4 sequences
bench_seqs  = (base_seqs * (BENCH_BATCH // len(base_seqs) + 1))[:BENCH_BATCH]

# Warm-up pass
run_inference(bench_seqs[:10])

# Timed pass
start = time.perf_counter()
bench_probs, _ = run_inference(bench_seqs)
elapsed = time.perf_counter() - start

tps = BENCH_BATCH / elapsed

print(f'Batch size          : {BENCH_BATCH} sequences  ({BENCH_BATCH * TIMESTEPS} transactions)')
print(f'Inference time      : {elapsed*1000:.1f} ms')
print(f'Throughput          : {tps:,.0f} sequences / second')
print(f'Active EP           : {session.get_providers()[0]}')